# Read From Bronze Table

In [0]:
%python
df = spark.table("databricks_bootcamp_dwb.bronze.crm_cust_info")
df.display()

# Data Transformations

## Duplicates

In [0]:
# Check for duplicate rows
duplicate_rows = df.groupBy(df.columns).count().filter("count > 1")
display(duplicate_rows)

In [0]:
from pyspark.sql.functions import monotonically_increasing_id
from pyspark.sql.functions import countDistinct

# Add a temporary index column to identify duplicate rows
df = df.withColumn("tmp_index", monotonically_increasing_id())

# Check for duplicate keys
dup_cst_key = df.groupBy("cst_key").count().filter("count > 1")
dup_rows_cst_key = df.join(dup_cst_key.select("cst_key"), on="cst_key", how="inner")
display(dup_rows_cst_key)

# Check that cst_key and cst_id are one-to-one
# Check for cst_key mapping to multiple cst_id
cst_key_multi_cst_id = df.groupBy("cst_key").agg(countDistinct("cst_id").alias("num_cst_id")).filter("num_cst_id > 1")
display(cst_key_multi_cst_id)

# Check for cst_id mapping to multiple cst_key
cst_id_multi_cst_key = df.groupBy("cst_id").agg(countDistinct("cst_key").alias("num_cst_key")).filter("num_cst_key > 1")
display(cst_id_multi_cst_key)

In [0]:
### Check that there is exactly one complete record for each cst_key

from pyspark.sql.functions import col

# STEP 1: Count distinct cst_keys across all rows
total_distinct_keys = dup_rows_cst_key.select("cst_key").distinct().count()

# STEP 2: Filter to rows with no nulls in any column
cols = dup_rows_cst_key.columns
clean_dup_rows = dup_rows_cst_key.dropna(how="any", subset=cols)

# STEP 2a: Count distinct cst_keys in null-free rows
clean_distinct_keys = clean_dup_rows.select("cst_key").distinct().count()

# STEP 2b: Check for duplicate cst_key in clean rows
clean_key_counts = clean_dup_rows.groupBy("cst_key").count().filter("count > 1")
clean_has_duplicates = clean_key_counts.count() > 0

# STEP 3: Evaluate success and show mismatch rows if failure
if not clean_has_duplicates and clean_distinct_keys == total_distinct_keys:
    result_status = "SUCCESS"
    mismatch_rows = spark.createDataFrame([], dup_rows_cst_key.schema)
else:
    mismatch_cst_keys = dup_rows_cst_key.select("cst_key").distinct().subtract(clean_dup_rows.select("cst_key").distinct())
    mismatch_rows = dup_rows_cst_key.join(mismatch_cst_keys, on="cst_key", how="inner")
    result_status = "FAILURE"
    display(mismatch_rows)
display(result_status)
print("\n", dup_rows_cst_key.count())
print(total_distinct_keys)

We have some users with trully incomplete information. 

In this case I am not sure which data to use for cst_create_date. I am worried that if I choose the row containing the later of the two dates and a customer placed an order before this date then I will create a logical inconsistancy. This logic could also be applied to the other customers with partial account creation and I need a deeper understanding of the account creation process before I can make a judgement on how to handle this. 

For now I will keep the most complete reccord and will look for logical inconsistencies down streem.

flag this for review in README.md

In [0]:
### Go through duplicates and chose the row with the least number of nulls to keep and drop the rest. 

from pyspark.sql.functions import sum, when
from functools import reduce
from operator import add
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Calculate number of nulls in each row of dup_rows_cst_key
null_count_expr = reduce(add, [when(col(c).isNull(), 1).otherwise(0) for c in dup_rows_cst_key.columns])
dup_cst_key_nulls = dup_rows_cst_key.withColumn("num_nulls", null_count_expr)

# Window specification to keep row with the fewest nulls (break ties with tmp_index)
window_spec = Window.partitionBy("cst_key").orderBy("num_nulls", "tmp_index")
dup_cst_key_nulls = dup_cst_key_nulls.withColumn("keep", (row_number().over(window_spec) == 1))

# Identify rows to drop based on tmp_index where keep == False
remove_tmp_index_values = dup_cst_key_nulls.filter(~col("keep")).select("tmp_index")

# Remove duplicate rows by anti-join on tmp_index
df_no_dup = df.join(remove_tmp_index_values, on="tmp_index", how="left_anti")

display(df_no_dup, "/n")
print(f"Rows after dropping duplicates: {df_no_dup.count()}")
print(f"Original rows: {df.count()}")

## Nulls

In [0]:
from pyspark.sql.functions import sum, col

# View nulls by column
null_counts = df_no_dup.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_no_dup.columns])
display(null_counts)

In [0]:
from functools import reduce

# Define columns to check for nulls (exclude cst_key and tmp_index)
columns_to_check = [c for c in df_no_dup.columns if c not in ['cst_key', 'tmp_index']]

# Create condition: all columns (except cst_key and tmp_index) are null
all_null_condition = reduce(lambda a, b: a & b, [col(c).isNull() for c in columns_to_check])

# Show rows that will be dropped (all columns except keys are null)
rows_to_drop = df_no_dup.filter(all_null_condition)
print(f"Rows to drop (all columns except cst_key and tmp_index are null): {rows_to_drop.count()}")
display(rows_to_drop)

In [0]:
# Filter out rows where all columns (except cst_key and tmp_index) are null
df_null_clean = df_no_dup.filter(~all_null_condition)

print(f"Rows after dropping all-null rows: {df_null_clean.count()}")
print(f"Rows before: {df_no_dup.count()}")
display(df_null_clean)

# Verify null counts after cleaning
null_counts = df_null_clean.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_null_clean.columns])
display(null_counts)

In [0]:
from pyspark.sql.functions import col, when

# Convert null values in cst_gndr to "n/a"
df_null_clean = df_null_clean.withColumn(
    "cst_gndr",
    when(col("cst_gndr").isNull(), "n/a").otherwise(col("cst_gndr"))
)

display(df_null_clean)

## Validate string values 
Check extra spaces, Identify abbreviations to normalize

In [0]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# trim strings
for field in df_null_clean.schema.fields:
    if isinstance(field.dataType, StringType):
        df_null_clean = df_null_clean.withColumn(field.name, trim(col(field.name)))

display(df_null_clean)

In [0]:
from pyspark.sql.functions import when

# normalize maritual status
df_null_clean = df_null_clean.withColumn(
    "cst_marital_status",
    when(col("cst_marital_status") == "S", "single")
    .when(col("cst_marital_status") == "M", "married")
    .otherwise(col("cst_marital_status"))
)

# normalize cst_gndr
df_null_clean = df_null_clean.withColumn(
    "cst_gndr",
    when(col("cst_gndr") == "F", "female")
    .when(col("cst_gndr") == "M", "male")
    .otherwise(col("cst_gndr"))
)

display(df_null_clean)

In [0]:
# make friendly column names
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "firstname",
    "cst_lastname": "lastname",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "date_created",
    "tmp_index": "tmp_index",
}

df_str_clean = df_null_clean.select([col(c).alias(RENAME_MAP.get(c, c)) for c in df_null_clean.columns])
display(df_str_clean)

## Validate Dates and Numeric

In [0]:
from pyspark.sql.types import IntegerType, DateType, StringType
from pyspark.sql.functions import col

# Define target data types for each column
type_mappings = {
    "customer_id": IntegerType(),
    "date_created": DateType(),
    "tmp_index": None  # Keep tmp_index as-is (LongType)
}

# Apply type casting
for field in df_str_clean.schema.fields:
    column_name = field.name
    
    if column_name in type_mappings:
        target_type = type_mappings[column_name]
        if target_type is not None:
            df_str_clean = df_str_clean.withColumn(column_name, col(column_name).cast(target_type))
    else:
        # All other columns should be StringType
        df_str_clean = df_str_clean.withColumn(column_name, col(column_name).cast(StringType()))

print("Data types after enforcement:")
schema_info = [(field.name, str(field.dataType)) for field in df_str_clean.schema.fields]
display(schema_info)

In [0]:
df_clean = df_str_clean.drop("tmp_index")
display(df_clean)

# Write Into Silver Table

In [0]:
df_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("databricks_bootcamp_dwb.silver.crm_customers")

In [0]:
%sql
SELECT * FROM databricks_bootcamp_dwb.silver.crm_customers